# box

> box_recipe() folds every SERVICES entry plus WireGuard and the maintenance baseline into one cloud-init YAML — plus the live rebuild flow

In [ ]:
#| default_exp box

## `box_recipe`

In [ ]:
#| export
from hetznerinit.core import cloud_init_dict, cloud_init_yaml
from hetznerinit.wireguard import wg_server_cmds
from hetznerinit.caddy import caddyfile, caddy_cmds
from boxrecipe.services import check_service, service_site
from boxrecipe.discopipe import discopipe_service
from boxrecipe.reconcile import reconcile_service
from boxrecipe.habitrack import habitrack_service

def box_recipe(
    services:list,             # service dicts (see boxrecipe.services)
    peers:dict,                # {"e15": {"ip": "10.0.0.2", "pubkey": "..."}}
    ssh_key:str,               # SSH public key line for the login user
    hostname:str="doyu-box",   # cloud-init hostname
)->str:                        # full cloud-init YAML for the box
    """Compose the box's cloud-init: every service + WireGuard + a maintenance baseline."""
    if not isinstance(ssh_key, str) or not ssh_key.strip():
        raise ValueError("ssh_key must be a non-empty string")
    services = [check_service(s) for s in services]
    ports = [s["port"] for s in services if s.get("port") is not None]
    if len(ports) != len(set(ports)):
        raise ValueError(f"service ports must be unique, got {sorted(ports)}")
    sites = [b for b in (service_site(s) for s in services) if b]
    caddy = caddy_cmds(caddyfile(sites)) if sites else []  # no sites → skip Caddy entirely
    packages = ["wireguard", "byobu", "curl", "git",
                "debian-keyring", "debian-archive-keyring", "apt-transport-https"]
    for s in services:
        for p in s.get("packages", []):
            if p not in packages:
                packages.append(p)
    svc_cmds = [c for s in services for c in s.get("cmds", [])]
    return cloud_init_yaml(cloud_init_dict(
        hostname, "doyu", ssh_key,
        packages=packages,
        cmds=[*wg_server_cmds(peers), *caddy, *svc_cmds],
        udp_ports=51820,
    ))

In [ ]:
DEMO_PEERS = {"e15": {"ip": "10.0.0.2", "pubkey": "A" * 43 + "="}}
DEMO_KEY = "ssh-ed25519 AAAADEMO doyu@example"
_RECONCILE = {"name": "reconcile", "domain": "reconcile.ninjalabo.ai",
              "port": 5001, "public": False, "packages": [], "cmds": []}

y = box_recipe([_RECONCILE, discopipe_service()], DEMO_PEERS, DEMO_KEY)
assert y.startswith("#cloud-config\n")
# maintenance baseline + wireguard (unchanged from the old hetznerinit deploy)
assert "- byobu" in y and "- wireguard" in y and "- ufw" in y
assert "ufw allow 51820/udp" in y
assert "A" * 43 + "=" in y
# per-service pieces folded in
assert "reconcile.ninjalabo.ai {" in y
assert "@vpn remote_ip 10.0.0.0/24" in y
assert "systemctl reload caddy" in y
assert "- python3-venv" in y
assert "/etc/systemd/system/discopipe.service" in y
assert "User=discopipe" in y and "ProtectSystem=strict" in y
assert "systemctl enable discopipe" in y
# no secret names/values ever reach user_data
for name in ("DISCORD_TOKEN", "DISCOPIPE_USER_ID", "DISCOPIPE_CHANNEL_ID", "ANTHROPIC_API_KEY"):
    assert name not in y, name

# public=True drops the gate and the 403 fallback for that site
y_pub = box_recipe([dict(_RECONCILE, public=True)], DEMO_PEERS, DEMO_KEY)
assert "@vpn" not in y_pub and "respond 403" not in y_pub
assert "reverse_proxy 127.0.0.1:5001" in y_pub

# domain=None services contribute no Caddy site; no sites → Caddy not even installed
y_d = box_recipe([discopipe_service()], DEMO_PEERS, DEMO_KEY)
assert "reverse_proxy" not in y_d
assert "apt-get install -y caddy" not in y_d

# port collisions are rejected
try:
    box_recipe([_RECONCILE, dict(_RECONCILE, name="other")], DEMO_PEERS, DEMO_KEY)
    assert False, "duplicate ports must raise"
except ValueError: pass

# ssh_key validation (unchanged from the old deploy)
for bad in ("", "   ", None):
    try:
        box_recipe([_RECONCILE], DEMO_PEERS, bad); assert False, f"must raise: {bad!r}"
    except ValueError: pass

## SERVICES

The registry: everything that runs on doyu-box. Adding a service = one
entry here (plus its fragment notebook if it needs install cmds).
Maturity path: flip `public` to True, rebuild.

In [ ]:
#| export
SERVICES = [reconcile_service(), discopipe_service(), habitrack_service()]

In [ ]:
assert [s["name"] for s in SERVICES] == ["reconcile", "discopipe", "habitrack"]
for s in SERVICES:
    check_service(s)
# the reconcile entry now carries its install+unit cmds (02_reconcile fragment)
assert any("/etc/systemd/system/reconcile.service" in c for c in SERVICES[0]["cmds"])
# habitrack (03_habitrack fragment): public site on its own port, full install cmds
assert any("/etc/systemd/system/habitrack.service" in c for c in SERVICES[2]["cmds"])
assert SERVICES[2]["port"] != SERVICES[0]["port"]

## Peers

Committed device list (public keys only — config, not secret). Kept out of the
published docs with `#| hide`. Seed the real pubkeys once from the running box:
`ssh doyu@box.ninjalabo.ai sudo cat /etc/wireguard/wg0.conf` (read each [Peer]'s
`# name` + AllowedIPs so e15↔10.0.0.2 / phone↔10.0.0.3 is not transposed).

In [ ]:
#| hide
PEERS = {
    "e15":   {"ip": "10.0.0.2", "pubkey": "awFS9HoxtslvUcFQ3fNOTPFUFl3F9N4Z9Jq+59xmbWM="},   # laptop
    "phone": {"ip": "10.0.0.3", "pubkey": "aoj3tHT4it+cHmZhhIyY6n20qp9wjNsilryS7dy701M="},   # phone
}

## Dry run

Build the full recipe without touching the API.

In [ ]:
DEMO = {"e15": {"ip": "10.0.0.2", "pubkey": "A" * 43 + "="}}
print(box_recipe(SERVICES, DEMO, "ssh-ed25519 AAAADEMO doyu@example"))

#cloud-config
hostname: doyu-box
preserve_hostname: false
package_update: true
package_upgrade: true
disable_root: true
ssh_pwauth: false
users:
- name: doyu
  groups:
  - sudo
  shell: /bin/bash
  sudo:
  - ALL=(ALL) NOPASSWD:ALL
  ssh_authorized_keys:
  - ssh-ed25519 AAAADEMO doyu@example
packages:
- ufw
- wireguard
- byobu
- curl
- git
- debian-keyring
- debian-archive-keyring
- apt-transport-https
- python3-venv
runcmd:
- ufw default deny incoming
- ufw default allow outgoing
- ufw allow 22/tcp
- ufw allow 80/tcp
- ufw allow 443/tcp
- ufw allow 51820/udp
- ufw --force enable
- umask 077 && wg genkey | tee /etc/wireguard/server.key | wg pubkey > /etc/wireguard/server.pub
- printf "[Interface]\nAddress = 10.0.0.1/24\nListenPort = 51820\nPrivateKey = " > /etc/wireguard/wg0.conf
- cat /etc/wireguard/server.key >> /etc/wireguard/wg0.conf
- printf "\n[Peer]\n# e15\nPublicKey = AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA=\nAllowedIPs = 10.0.0.2/32\n" >> /etc/wireguard/wg0.conf
- chmod 600

## Live run: rebuild the box

All cells below are `#| eval: false`. Run in order. This DESTROYS and recreates
doyu-box. Fill `PEERS` with real pubkeys first. If the new IP differs, update the
GoDaddy wildcard `*` A record (see the hetzner-rebuild skill) before the client
steps.

In [ ]:
#| eval: false
import os
from pathlib import Path
from hetznerinit.server import (get_client, server_spec, check_cloud_config,
                                create_server, delete_server, wait_running, check_cloud_init)
from hetznerinit.wireguard import get_server_pubkey

assert all(p["pubkey"] != "..." for p in PEERS.values()), "fill PEERS first"

def _ssh_pubkey():
    env = os.environ.get("SSH_PUBKEY_PATH")
    cands = [Path(env)] if env else [Path.home()/".ssh"/n for n in ("id_ed25519.pub", "id_rsa.pub")]
    for p in cands:
        if p.exists(): return p.read_text().strip()
    raise FileNotFoundError(f"no SSH public key found (tried {[str(c) for c in cands]}); set SSH_PUBKEY_PATH")

ssh_key = _ssh_pubkey()
user_data = box_recipe(SERVICES, PEERS, ssh_key)
print(check_cloud_config(user_data))
print(user_data)  # inspect the REAL user_data before the create cell

Valid schema /tmp/tmp0qwdkjqu.yaml
#cloud-config
hostname: doyu-box
preserve_hostname: false
package_update: true
package_upgrade: true
disable_root: true
ssh_pwauth: false
users:
- name: doyu
  groups:
  - sudo
  shell: /bin/bash
  sudo:
  - ALL=(ALL) NOPASSWD:ALL
  ssh_authorized_keys:
  - ssh-rsa AAAAB3NzaC1yc2EAAAADAQABAAABgQDTjk9TD98vjRrBOUakASlbBMatM50JHZls0dx4D6aa01H3kNCqQjMYv9T/LPyzoZYoMo2LOWNM8gOTdB0nJDyKXhGJIZ3WtIvj+kIYIhsRQTtqZ4t8TPrHZ7D4s95cfuEpmYV7M9ZAqJJUMVEEkTJrcDhgL4TSv8sP5gLGqd/CFHUY/jEfw3okGg5mzpcMIw3A+RoBjQFbk9pE17Ividjlgpx8eqac/cJY0V9Hp3KMWGJmbahNtVi/nWaH4bOCD8P9ofGZ1Y8mbxsIR4SrPE+CEvJOPFKcCP3IWLRNxqdCA8ajeShh0XogqvKb+WgOukynnZZl6dmeJg3uliQCCiJR8goBmTKUsOrfQcs3w7bqaRAo2pzeycCESiB5ZWZTZNRBK4JHYd8/nMFXKJfDjnAbIc9B1DtGWIGwbZrXFPGfOo9arJWbHXqZthvRoqPmP1cO10uvVJhxqTdhscPt2YM0PQJdas5KCr/Ydy/zu58Szr9gT5XAKmQUwoFD7u9HLo8= doyu@e15
packages:
- ufw
- wireguard
- byobu
- curl
- git
- debian-keyring
- debian-archive-keyring
- apt-transport-https
- python3-venv
runcmd:
- ufw def

In [ ]:
#| eval: false
client = get_client()
old = client.servers.get_by_name("doyu-box")
if old is not None:
    delete_server(old, confirm=True)
    print(f"deleted old box id={old.id}")
srv = create_server(client, server_spec("doyu-box", user_data), confirm=True)
print(f"created: id={srv.id} name={srv.name} ip={srv.public_net.ipv4.ip}")
print("cleanup if anything below fails:")
print("  srv = client.servers.get_by_name('doyu-box'); delete_server(srv, confirm=True)")

deleted old box id=148844879


created: id=151641945 name=doyu-box ip=77.42.89.53
cleanup if anything below fails:
  srv = client.servers.get_by_name('doyu-box'); delete_server(srv, confirm=True)


In [ ]:
#| eval: false
import subprocess as _sp
ip = wait_running(srv)
for target in (ip, "box.ninjalabo.ai"):
    _sp.run(["ssh-keygen", "-R", target], capture_output=True)
dns_ip = _sp.run(["dig", "+short", "box.ninjalabo.ai"], capture_output=True, text=True).stdout.strip()
print(f"box ip: {ip}  /  DNS says: {dns_ip}")
if dns_ip != ip:
    print(f"UPDATE GoDaddy: set the wildcard '*' A record to {ip}, wait for TTL, then continue")

box ip: 77.42.89.53  /  DNS says: 157.180.75.135
UPDATE GoDaddy: set the wildcard '*' A record to 77.42.89.53, wait for TTL, then continue


In [ ]:
#| eval: false
print(check_cloud_init(ip, "doyu"))

In [ ]:
#| eval: false
# verify Caddy and the discopipe unit (waiting on its env file is expected here)
import subprocess as _sp
def _ssh(*remote):
    return _sp.run(["ssh", "-o", "BatchMode=yes", "-o", "StrictHostKeyChecking=accept-new",
                    "doyu@" + ip, *remote], capture_output=True, text=True).stdout.strip()
print("caddy active:", _ssh("systemctl", "is-active", "caddy"))
print("config valid:", _ssh("sudo", "caddy", "validate", "--config", "/etc/caddy/Caddyfile"))
print("discopipe enabled:", _ssh("systemctl", "is-enabled", "discopipe"))
print("discopipe active (inactive until env file lands):", _ssh("systemctl", "is-active", "discopipe"))

## Install the discopipe env file (manual, secrets)

Same rank as the GoDaddy DNS step: user_data never carries secrets, so the
env file is installed by hand after boot. **Port** the laptop's values —
`DISCOPIPE_CWD` changes to `/home/discopipe/agent`, `ANTHROPIC_API_KEY` is
added (design Decision 3). Run on the laptop:

```sh
umask 077
cp ~/.config/discopipe/env /tmp/box-env
sed -i 's|^DISCOPIPE_CWD=.*|DISCOPIPE_CWD=/home/discopipe/agent|' /tmp/box-env
read -rs ANTHROPIC_API_KEY   # paste the key: no echo, never in shell history
printf 'ANTHROPIC_API_KEY=%s\n' "$ANTHROPIC_API_KEY" >> /tmp/box-env
unset ANTHROPIC_API_KEY
scp /tmp/box-env doyu@<IP>:/tmp/env && rm /tmp/box-env
ssh doyu@<IP> 'sudo install -m 600 -o root -g root /tmp/env /etc/discopipe/env && rm /tmp/env \
  && sudo systemctl start discopipe && systemctl is-active discopipe'
```

Then send one Discord message in the bot's channel and confirm a reply
(the laptop bot is already stopped — `systemctl --user is-enabled discopipe`
must report disabled).

In [ ]:
#| eval: false
# server key is regenerated on every rebuild → refresh the SERVER pubkey in each client
spub = get_server_pubkey(ip, "doyu")
print(f"new server pubkey: {spub}")
print("laptop: update /etc/wireguard/box.conf then restart the tunnel:")
print(f"  sudo sed -i 's|PublicKey = .*|PublicKey = {spub}|' /etc/wireguard/box.conf")
print("  sudo wg-quick down box && sudo wg-quick up box")
print("phone: in the WireGuard app, set the peer's Public key to the value above")

In [ ]:
#| eval: false
# after the laptop tunnel is back up
import subprocess as _sp
print(_sp.run(["ping", "-c", "3", "10.0.0.1"], capture_output=True, text=True).stdout)
print(_sp.run(["ssh", "-o", "BatchMode=yes", "-o", "StrictHostKeyChecking=accept-new",
               "doyu@10.0.0.1", "hostname"], capture_output=True, text=True).stdout)

## Add a new WireGuard client (only when needed)

The ONLY place a client keypair is generated. Off the normal rebuild path.

In [ ]:
#| eval: false
from hetznerinit.wireguard import wg_keypair, wg_client_conf, get_server_pubkey
import subprocess as _sp

name, ip_addr = "newdev", "10.0.0.4"          # pick an unused name + 10.0.0.x
spub = get_server_pubkey("box.ninjalabo.ai", "doyu")
priv, pub = wg_keypair()
print(f'add to PEERS, commit, then rebuild:  "{name}": {{"ip": "{ip_addr}", "pubkey": "{pub}"}}')
conf = wg_client_conf(name, priv, spub, ip_addr)
# the QR encodes this client's PRIVATE key — scan it on the phone, then CLEAR
# this cell's output before saving/committing (do not leave it in the notebook)
_sp.run(["qrencode", "-t", "ansiutf8"], input=conf, text=True)

## Client access (per VPN device)

Every `public=False` service with a domain needs a client-side split-DNS
override: an `/etc/hosts` line `10.0.0.1 <domain>` on each VPN device,
or Caddy's `remote_ip` gate returns 403 (traffic to the public IP
bypasses the tunnel). Verify per domain:
`curl --resolve <domain>:443:10.0.0.1 https://<domain>`.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()